# TF-IDF et sentiment : lire les poids avant de prédire

**Objectifs observables (75–100 min).** Montrer comment un terme devient une colonne, calculer une pondération TF-IDF sur trois phrases, tracer sa matrice, entraîner un baseline et une pipeline sans fuite, puis inspecter les erreurs.

**Prérequis :** split, métriques de classification, fonctions Python. Le corpus ci-dessous a été écrit pour ce cours : il est très petit, en anglais et artificiel. Son score ne représente aucune population réelle.

## 1. De la phrase aux features

Une phrase est une suite de caractères. `TfidfVectorizer` découpe cette chaîne en tokens, compte les termes, puis donne un poids plus grand aux termes fréquents dans un document mais moins présents ailleurs. Avec le lissage scikit-learn, $idf(t)=\log[(1+n)/(1+df(t))]+1$. `n` est le nombre de documents d'entraînement, `df(t)` le nombre de documents contenant le terme. Une normalisation L2 est appliquée à chaque ligne par défaut.

Le mot « service » commun à tous les documents porte moins d'information discriminante que « rude », présent dans un seul. TF-IDF ne comprend ni le sens, ni l'ironie, ni un synonyme absent.

In [ ]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

texts = [
 'excellent service and kind staff', 'the meal was fresh and delicious', 'fast delivery and great quality', 'friendly team solved my issue',
 'wonderful experience and helpful support', 'the product works perfectly', 'clear instructions and easy setup', 'I am happy with this purchase',
 'terrible service and rude staff', 'the meal was cold and tasteless', 'slow delivery and poor quality', 'unhelpful team ignored my issue',
 'awful experience and frustrating support', 'the product broke immediately', 'confusing instructions and difficult setup', 'I regret this purchase',
 'great quality but the box was damaged', 'helpful support but delivery was slow', 'fresh food and friendly staff', 'poor product and confusing manual',
 'fast shipping and perfect item', 'rude response and long wait', 'kind team and useful solution', 'cold food arrived late'
]
labels = [1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,1,0,1,0,1,0,1,0]
frame = pd.DataFrame({'text': texts, 'sentiment': labels})
print(frame.groupby('sentiment').size())
frame.head()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

toy_docs = ['kind service', 'rude service', 'kind staff service']
toy_vectorizer = TfidfVectorizer(norm=None, use_idf=True)
toy_matrix = toy_vectorizer.fit_transform(toy_docs).toarray()
toy_terms = toy_vectorizer.get_feature_names_out()
fig, ax = plt.subplots(figsize=(7, 3))
image = ax.imshow(toy_matrix, cmap='Blues', vmin=0)
ax.set_xticks(range(len(toy_terms)), toy_terms)
ax.set_yticks(range(len(toy_docs)), toy_docs)
for row_index in range(len(toy_docs)):
    for term_index in range(len(toy_terms)):
        ax.text(term_index, row_index, f'{toy_matrix[row_index, term_index]:.2f}',
                ha='center', va='center', color='black')
ax.set(title='Poids TF-IDF non normalisés sur trois documents', xlabel='Terme', ylabel='Document')
fig.colorbar(image, ax=ax, label='TF × IDF')
plt.tight_layout(); plt.show()
print('Termes et idf:', dict(zip(toy_terms, toy_vectorizer.idf_.round(3))))

## 2. Split avant le vocabulaire

`fit` du vectorizer apprend le vocabulaire et les statistiques `df` du train. Si l'on vectorise tout avant de séparer, les mots et fréquences du test contaminent les features. Une pipeline place `TfidfVectorizer` puis `LogisticRegression` et les réajuste ensemble dans chaque fold. L'output de TF-IDF est une matrice éparse `(documents, termes)`, utile quand la plupart des termes sont absents de chaque document.

In [ ]:
text_train, text_test, y_train, y_test = train_test_split(
    frame['text'], frame['sentiment'], test_size=0.25, stratify=frame['sentiment'], random_state=12
)
text_baseline = DummyClassifier(strategy='most_frequent').fit(text_train.to_frame(), y_train)
text_model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=1),
    LogisticRegression(C=1.0, max_iter=1_000)
)
text_model.fit(text_train, y_train)
prediction = text_model.predict(text_test)
print('majority baseline accuracy:', text_baseline.score(text_test.to_frame(), y_test))
print(classification_report(y_test, prediction, zero_division=0))
print('confusion matrix:', confusion_matrix(y_test, prediction).tolist())

## 3. Interpréter coefficients et erreurs

La régression logistique calcule un logit $z=w^T x+b$, puis $P(y=1|x)=1/(1+e^{-z})$. Les composantes de `x` sont les poids TF-IDF. Les coefficients positifs/négatifs indiquent une association dans **ce** train et pour **cette** régularisation ; ils ne décrivent pas une cause. Une expression négative contenant un mot positif peut dérouter ce modèle à courte portée.

In [ ]:
vectorizer = text_model.named_steps['tfidfvectorizer']
classifier = text_model.named_steps['logisticregression']
weights = pd.Series(classifier.coef_[0], index=vectorizer.get_feature_names_out())
print('termes positifs associés :\n', weights.nlargest(8).round(2))
print('termes négatifs associés :\n', weights.nsmallest(8).round(2))
error_review = pd.DataFrame({'text': text_test, 'actual': y_test, 'predicted': prediction})
error_review[error_review['actual'] != error_review['predicted']]

In [ ]:
selected_terms = pd.concat([weights.nsmallest(6), weights.nlargest(6)]).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(selected_terms.index, selected_terms.values,
        color=['#ad4e3c' if value < 0 else '#326d84' for value in selected_terms.values])
ax.axvline(0, color='black', linewidth=0.7)
ax.set(xlabel='Coefficient du logit pour la classe positive',
       title='Termes associés aux prédictions dans ce petit train')
plt.tight_layout(); plt.show()

## 4. Expérience, limites et pratique

Comparer unigrammes seuls à unigrammes + bigrammes avec validation croisée sur le train. Le test se lit une seule fois une fois le choix arrêté. Un texte multilingue demande une tokenisation et une évaluation adaptées ; une liste de stop words naïve peut supprimer des négations importantes.

**Exercices.** Calculez à la main `df` de « service » et « rude » dans le jouet ; changez une phrase pour « not good » et expliquez pourquoi les bigrammes peuvent aider ; testez des paraphrases hors du corpus et documentez les échecs. Embeddings, RNN/LSTM, self-attention et Transformers nécessitent des leçons distinctes.